In [22]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn

In [12]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [13]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [14]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['diagnosis']), df['diagnosis'], test_size=0.2, random_state=42)

In [15]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [16]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [17]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [18]:
class MySimpleNN(): 

    def __init__(self, X): 
        self.weights = torch.rand(X.shape[1], 1, dtype = torch.float64, requires_grad = True)
        self.bias = torch.zeros(1, dtype = torch.float64, requires_grad = True)
    
    def forward(self, X):
        z = torch.matmul(X, self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred
    
    def loss_function(self, y_pred, y):
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)
        loss = -(y*torch.log(y_pred) + (1 - y)*torch.log(1 - y_pred)).mean()
        return loss


In [19]:
learning_rate = 0.1
epochs = 25


In [20]:
# Defining model 
model = MySimpleNN(X_train_tensor)


# define loop

for epoch in range(epochs):

    # forward pass
    y_pred = model.forward(X_train_tensor)

    # loss
    loss = model.loss_function(y_pred, y_train_tensor)


    # backward pass
    loss.backward()


    # parameters update
    with torch.no_grad():
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

    # zero gradients
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    # print loss in each epoch
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item()}")


Epoch 1/25, Loss: 3.499076569765995
Epoch 2/25, Loss: 3.37301589199679
Epoch 3/25, Loss: 3.244263618697993
Epoch 4/25, Loss: 3.1092921483251468
Epoch 5/25, Loss: 2.9700811650166177
Epoch 6/25, Loss: 2.8230030063253198
Epoch 7/25, Loss: 2.6719943273220106
Epoch 8/25, Loss: 2.5190804371742677
Epoch 9/25, Loss: 2.3684627023691704
Epoch 10/25, Loss: 2.2197167615804285
Epoch 11/25, Loss: 2.0718918073682318
Epoch 12/25, Loss: 1.9211372208510096
Epoch 13/25, Loss: 1.7766017546653339
Epoch 14/25, Loss: 1.6384535480452567
Epoch 15/25, Loss: 1.5027503604391563
Epoch 16/25, Loss: 1.3806755810856062
Epoch 17/25, Loss: 1.2734434657751845
Epoch 18/25, Loss: 1.1816135200577396
Epoch 19/25, Loss: 1.1049221953665282
Epoch 20/25, Loss: 1.042302117104444
Epoch 21/25, Loss: 0.9920457830780948
Epoch 22/25, Loss: 0.9520664936308988
Epoch 23/25, Loss: 0.9202032297024213
Epoch 24/25, Loss: 0.8944915896918371
Epoch 25/25, Loss: 0.8733317984566877


In [23]:
class Model(nn.Module): 

    def __init__(self, num_features): 

        super().__init__()

        self.linear = nn.Linear(num_features,1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, features): 
        out = self.linear(features)
        out = self.sigmoid(out) 
        return out 
    


In [26]:
features = torch.rand(10,5)

model = Model(features.shape[1])

In [29]:
model.linear.bias

Parameter containing:
tensor([0.2131], requires_grad=True)